In [ ]:
import numpy as np
import pandas as pd
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [ ]:
transform = transforms.Compose([
    transforms.Resize((28,28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

In [ ]:
trainDataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainDataloader = DataLoader(trainDataset, batch_size=64, shuffle=True)

In [ ]:
testDataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testDataloader = DataLoader(testDataset, batch_size=64, shuffle=True)

In [ ]:
class CnnModel(nn.Module):
  def __init__(self, outpitSize):
    super().__init__()

    self.Model = nn.Sequential(
        nn.Conv2d(1,16,2,1,2),
        nn.MaxPool2d(2,2),
        nn.ReLU(),

        nn.Conv2d(16,32,2,1,2),
        nn.MaxPool2d(2,2),
        nn.ReLU(),

        nn.Conv2d(32,64,2,1,2),
        nn.MaxPool2d(2,2),
        nn.ReLU(),

        nn.Conv2d(64,128,2,1,2),
        nn.MaxPool2d(2,2),
        nn.ReLU(),

        nn.Flatten(),
        nn.Linear(128 * 4 * 4, outpitSize)
    )

  def forward(self, x):
    return self.Model(x)

In [ ]:
cnnModel = CnnModel(10).to(device)

In [ ]:
criteria = nn.CrossEntropyLoss()

In [ ]:
cnnOptimizer = torch.optim.Adam(cnnModel.parameters(), lr=0.001)

In [ ]:
epochs = 10

In [ ]:
for epoch in range(epochs):
  for image, label in trainDataloader:

    image = image.to(device)
    label = label.to(device)

    op = cnnModel(image)
    loss = criteria(op, label)

    cnnOptimizer.zero_grad()
    loss.backward()
    cnnOptimizer.step()

  print(f'Epoch: {epoch+1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.032261718064546585
Epoch: 2, Loss: 0.060091838240623474
Epoch: 3, Loss: 0.03929783031344414
Epoch: 4, Loss: 0.16898968815803528
Epoch: 5, Loss: 0.0019856784492731094
Epoch: 6, Loss: 0.0007556484779343009
Epoch: 7, Loss: 0.0544588640332222
Epoch: 8, Loss: 0.11032592505216599
Epoch: 9, Loss: 0.04304088279604912
Epoch: 10, Loss: 0.0004960557562299073


In [ ]:
cnnModel.eval()

correct = 0
total = 0

with torch.no_grad():
  for image, label in testDataloader:

    image = image.to(device)
    label = label.to(device)

    op = cnnModel(image)

    _, prediction = torch.max(op, 1)

    correct += (prediction == label).sum().item()
    total += label.size(0)

print(f'Accuracy: {100 * correct / total}%')

Accuracy: 99.04%
